In [58]:
# ============================================================
# Cellule 1 — Imports et chargement des fichiers
# ============================================================
import json
import csv
import re, os
from pathlib import Path


corpus = 'JJ160-JJ169'





JSON_PATH = f"../List-of-zones/Himanis_Seg_Actes_1200pxmin_{corpus}_labelstudio.json"   # <- ton export Label Studio
OUTPUT_PATH = f"../List-of-zones/Himanis_Seg_Actes_1200pxmin_{corpus}_labelstudio_completed.json"
CSV_PATH  = "../List-of-images/JJ096-JJ211_image_data_with_resolved_urls.csv"                # <- ta liste complète d'images



def parse_corpus(corpus):
    match = re.match(r'([A-Z]+)(\d+)-([A-Z]+)(\d+)', corpus)
    if not match:
        raise ValueError(f"Format inattendu : {corpus}")

    prefix = match.group(1)
    start = int(match.group(2))
    end = int(match.group(4))
    width = len(match.group(2))  # conserve le nombre de chiffres initial

    return {f"{prefix}{i:0{width}d}" for i in range(start, end + 1)}


REGISTRES_CIBLES = sorted(parse_corpus(corpus))
print(REGISTRES_CIBLES)



with open(JSON_PATH, "r", encoding="utf-8") as f:
    tasks = json.load(f)

print(f"Nombre de tâches dans le JSON : {len(tasks)}")
print("Exemple de tâche :")
print(json.dumps(tasks[0], indent=2, ensure_ascii=False)[:1000])

# ============================================================
# Cellule 2 — Repérer le champ qui contient le nom/chemin de l'image
# ============================================================
# Dans un export Label Studio classique, chaque tâche a la forme :
# { "id": ..., "data": {"image": "..."}, "annotations": [...], ... }
# Adapte le nom de la clé si besoin (ex: "image", "img", "ocr", etc.)

IMAGE_FIELD = "image_path"

print(tasks[0]["data"].keys())

['JJ160', 'JJ161', 'JJ162', 'JJ163', 'JJ164', 'JJ165', 'JJ166', 'JJ167', 'JJ168', 'JJ169']
Nombre de tâches dans le JSON : 5958
Exemple de tâche :
{
  "id": 17847,
  "annotations": [
    {
      "id": 17847,
      "completed_by": 1,
      "result": [],
      "was_cancelled": false,
      "ground_truth": false,
      "created_at": "2026-06-19T11:25:09.709867Z",
      "updated_at": "2026-06-19T11:25:09.709867Z",
      "draft_created_at": null,
      "lead_time": null,
      "prediction": {},
      "result_count": 0,
      "unique_id": "145f23e8-2afe-4a77-8dba-0ba88e712833",
      "import_id": 29133,
      "last_action": null,
      "bulk_created": false,
      "task": 17847,
      "project": 6,
      "updated_by": null,
      "parent_prediction": null,
      "parent_annotation": null,
      "last_created_by": null
    }
  ],
  "file_upload": "c9786b02-Himanis_Seg_Actes_1200pxmin_JJ160-JJ169_labelstudio.json",
  "drafts": [],
  "predictions": [],
  "data": {
    "image": "https://iiif.irh

In [59]:
from os.path import basename

import re

def numeric_key(name: str):
    filename = Path(name).name  # enlève le chemin/URL éventuel
    numbers = re.findall(r"\d+", filename)
    return [int(n) for n in numbers] if numbers else [float("inf")]


# ============================================================
# Cellule 3 (modifiée) — charger le CSV en dict pour accès par nom d'image
# ============================================================
CSV_IMAGE_COLUMN = "imageFileName" # images_registres_AN_JJ035_JJ211/images\Paris_Archives_Nationales_JJ096\Paris_Archives_Nationales_JJ096_1.jpg
CSV_URL_COLUMN = "urlImage_arkId"        # https://iiif.irht.cnrs.fr/iiif/ark:/63955/vd0qz1ihxyni/full/full/0/default.jpg

csv_rows_by_image = {}
with open(CSV_PATH, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        name = row[CSV_IMAGE_COLUMN].strip()
        if name:
            csv_rows_by_image[basename(name)] = row

csv_images = list(csv_rows_by_image.keys())

csv_images_sorted = sorted(
    [
        x for x in csv_images
        if any(registre in Path(x).name for registre in REGISTRES_CIBLES)
    ],
    key=numeric_key
)

In [60]:
# ============================================================
# Cellule 5 — Identifier les images manquantes dans le JSON
# ============================================================

def get_image_source(task: dict) -> str:
    """
    Retourne le chemin/nom de fichier de l'image.

    Priorité :
    1. meta["source_file"]  → anciennes tâches (JJ096-JJ139)
    2. data["image_path"]   → nouvelles tâches (JJ140-JJ211)
    """

    source_file = task.get("meta", {}).get("source_file")
    if source_file:
        return source_file.strip()

    image_path = task.get("data", {}).get("image_path")
    if image_path:
        return image_path.strip()

    return ""

def basename(path_or_url: str) -> str:
    """
    Extrait le nom de fichier en gérant / et \\
    """
    return re.split(r"[\\/]", path_or_url)[-1]


# Images présentes dans le JSON
existing_images = {
    basename(get_image_source(t))
    for t in tasks
    if get_image_source(t)
}

# Images du CSV qui ne sont pas présentes dans le JSON
missing_images = [
    img for img in csv_images_sorted
    if basename(img) not in existing_images
]

print(f"Images présentes dans le JSON : {len(existing_images)}")
print(f"Images manquantes ({len(missing_images)}) :")

for m in missing_images:
    print(" -", m)

Images présentes dans le JSON : 5958
Images manquantes (0) :


In [47]:
# ============================================================
# Cellule 6 — Créer les nouvelles tâches
# ============================================================

def make_new_task(image_key: str) -> dict:
    row = csv_rows_by_image[image_key]

    # --- valeur brute telle qu'elle apparaît dans le CSV ---
    full_image_path = row[CSV_IMAGE_COLUMN].strip()

    # --- champ "image" : URL avec remplacement de "/full/full/" ---
    url_image = row.get(CSV_URL_COLUMN, "").strip()
    image_value = url_image.replace("/full/full/", "/full/1200,/")

    # --- registre / ordre à partir du chemin complet ---
    stem = Path(full_image_path.replace("\\", "/")).stem
    tokens = stem.split("_")

    # Exemple :
    # Paris_Archives_Nationales_JJ118_536.jpg
    #
    # tokens[-2] = JJ118
    # tokens[-1] = 536

    registre = tokens[-2] if len(tokens) >= 2 else None
    ordre = tokens[-1] if len(tokens) >= 1 else None

    return {
        "id": None,
        "data": {
            "image": image_value,
            IMAGE_FIELD: full_image_path,
            "registre": registre,
            "ordre": ordre
        },
        "annotations": [],
        "predictions": []
    }


new_tasks = [make_new_task(img) for img in missing_images]

# ============================================================
# Cellule 7 — Fusionner et trier par registre puis par ordre
# ============================================================

all_tasks = tasks + new_tasks


def task_sort_key(task):
    """
    Tri :
        1. registre : JJ096, JJ097, JJ102, JJ118, etc.
        2. ordre    : 1, 2, 3, ..., 10, 11, ...
    """

    # --------------------------------------------------------
    # 1. Récupérer le source_file du JSON
    # --------------------------------------------------------
    source_file = task.get("meta", {}).get("source_file", "")

    # Pour les nouvelles tâches, source_file n'existe pas encore.
    # On utilise alors le image_path.
    if not source_file:
        source_file = task.get("data", {}).get(IMAGE_FIELD, "")

    # --------------------------------------------------------
    # 2. Extraire le nom de fichier
    # --------------------------------------------------------
    filename = re.split(r"[\\/]", source_file)[-1]

    # Exemple :
    # Paris_Archives_Nationales_JJ118_536.jpg
    #
    # -> JJ118
    # -> 536

    match = re.search(r"(JJ\d+)_(\d+)(?:\.[^.]+)?$", filename)

    if match:
        registre = match.group(1)
        ordre = int(match.group(2))
    else:
        # Fallback si le nom ne correspond pas au format attendu
        registre = "ZZ999"
        ordre = float("inf")

    # Extraire le numéro du registre
    registre_num = int(re.search(r"\d+", registre).group())

    return (registre_num, ordre)


all_tasks_sorted = sorted(
    all_tasks,
    key=task_sort_key
)

print(f"Total après fusion : {len(all_tasks_sorted)} tâches")

Total après fusion : 5228 tâches


In [48]:
# ============================================================
# Cellule 8 — Sauvegarder le nouveau JSON d'import
# ============================================================
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(all_tasks_sorted, f, ensure_ascii=False, indent=2)

print(f"Fichier écrit : {OUTPUT_PATH}")

Fichier écrit : ../List-of-zones/Himanis_Seg_Actes_1200pxmin_JJ119-JJ132_labelstudio_completed.json
